In [ ]:
# ========================
# 07_metrics_to_semantic_with_skeleton.ipynb
# 從六大指標數據反過來生成 LLM 語義對齊文字，並在旁邊附加動態骨架以便對照
# ========================
import pandas as pd
import json
import numpy as np
import os
from pathlib import Path
from openai import OpenAI
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display, clear_output

# OpenAI API Key 設定
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

print(f"OpenAI 模型已設定為：{OPENAI_MODEL}")

OpenAI 模型已設定為：gpt-4o-mini


In [2]:
# ========================
# Discord Multi-Agent Bridge Setup
# ========================
from dance_discord_utils import DiscordAgentBridge, AnalysisState
import time

# Initialize state manager for synchronized broadcasting
state_manager = AnalysisState()
state_manager.reset()

print("✅ Discord 即時串流橋接器與狀態管理器準備就緒！")

✅ Discord 即時串流橋接器準備就緒！


In [3]:
# 定義分析資料夾路徑
folder = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/data/bajiajiang_Analysis_Results/fusion/")

# 讀取指標資料與骨架資料
energy_df = pd.read_csv(folder / "energy.csv")
geometry_df = pd.read_csv(folder / "geometry.csv")
stability_df = pd.read_csv(folder / "stability.csv")
sync_df = pd.read_csv(folder / "synchronization.csv")
trans_df = pd.read_csv(folder / "transition.csv")
skeleton_df = pd.read_csv(folder / "fusion_skeleton.csv")

# 載入八家將文化資料庫 (Lookup Table)
cultural_lib_path = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/ba_jia_jiang_cultural_library.json")
with open(cultural_lib_path, "r", encoding="utf-8") as f:
    cultural_library = json.load(f)

print(f"✅ 指標與骨架資料載入成功！")
print(f"✅ 文化資料庫載入成功！共 {len(cultural_library)} 筆項目")

✅ 指標與骨架資料載入成功！
✅ 文化資料庫載入成功！共 76 筆項目


In [4]:
# 解析 skeleton.csv，將每幀的座標取出並轉換為 Numpy Array
num_frames = len(skeleton_df)
num_joints = 17
skel_data = np.zeros((num_frames, num_joints, 3))

for j in range(num_joints):
    col_str = skeleton_df[f'Joint_{j}']
    # 解析字串 'x, y, z' 到 float 陣列
    parsed = col_str.apply(lambda x: [float(v) for v in x.split(',')])
    skel_data[:, j, :] = np.vstack(parsed.values)

print(f"✅ 骨架資料解析完成！陣列形狀: {skel_data.shape} (Frames, Joints, XYZ)")

✅ 骨架資料解析完成！陣列形狀: (2174, 17, 3) (Frames, Joints, XYZ)


In [5]:
# 設定取樣間隔 (例如每 5 秒一個語義轉折點)
FPS = 30
INTERVAL_SEC = 5
INTERVAL_FRAMES = INTERVAL_SEC * FPS

total_frames = len(energy_df)
semantic_segments = []

for start_f in range(0, total_frames, INTERVAL_FRAMES):
    end_f = min(start_f + INTERVAL_FRAMES, total_frames)
    f_range = range(start_f, end_f)
    
    # 聚合這段時間的指標平均值
    seg_metrics = {
        'timestamp_sec': round(start_f / FPS, 2),
        'frame_start': start_f,
        'energy': energy_df.iloc[f_range]['energy'].mean(),
        'volume': geometry_df.iloc[f_range]['volume'].mean(),
        'curvature': geometry_df.iloc[f_range]['curvature'].mean(),
        'sway': stability_df.iloc[f_range]['sway'].mean(),
        'correlation': sync_df.iloc[f_range]['correlation'].mean(),
        'torque': trans_df.iloc[f_range]['torque'].mean(),
        'jerk': trans_df.iloc[f_range]['jerk'].mean()
    }
    semantic_segments.append(seg_metrics)

segments_df = pd.DataFrame(semantic_segments)
print(f"🔹 已切分為 {len(segments_df)} 個語義片段")

🔹 已切分為 15 個語義片段


In [6]:
# ========================
# Multi-Persona AI Logic (Sees) - Bajiajiang Version
# ========================

INTERVAL_SEC = 5.0

SYSTEM_PROMPT_SEES = """
你是一位專業的舞蹈分析專家。你的任務是根據提供的物理指標數據，對舞動進行客觀且技術性的描述。
請務必參考提供的「文化資料庫」，使用其中的 `term`（術語）與 `short_phrase`（短句）來描述動作。

【輸出格式】：
【AI sees】 [技術性描述]
【Keywords】 [本次描述中引用的術語，以逗號分隔]
"""

def safe_float(val):
    """確保數值為有效 float，避免 NaN/Inf 導致 API 錯誤"""
    try:
        f_val = float(val)
        if np.isnan(f_val) or np.isinf(f_val):
            return 0.0
        return f_val
    except:
        return 0.0

def ai_sees(metrics, library):
    e = safe_float(metrics.get('energy', 0))
    v = safe_float(metrics.get('volume', 0))
    t = safe_float(metrics.get('torque', 0))
    j = safe_float(metrics.get('jerk', 0))
    
    lib_ref = "\n".join([f"- {item.get('term', 'N/A')}: {item.get('short_phrase', 'N/A')}" for item in library])
    
    prompt = f"""
    當前分析指標：
    - Energy: {e:.2f}, Volume: {v:.4f}
    - Torque: {t:.2f}, Jerk: {j:.2f}
    
    參考資料庫：
    {lib_ref}
    """
    
    try:
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_SEES.strip()},
                {"role": "user", "content": prompt.strip()}
            ],
            temperature=0.3 
        )
        raw_output = response.choices[0].message.content.strip()
    except Exception as err:
        print(f"❌ AI Sees API 錯誤: {err}")
        return "【AI sees】 無法生成描述", "無"
    
    sees_part = "【AI sees】 無法解析描述"
    keywords_part = "無"
    for line in raw_output.split("\n"):
        if "【AI sees】" in line:
            sees_part = line
        elif "【Keywords】" in line:
            keywords_part = line.replace("【Keywords】", "").strip()
            
    return sees_part, keywords_part

print("✅ AI Sees 生成邏輯準備完成！")

✅ 三種八家將人格 AI Agents 生成邏輯準備完成！


In [7]:
def create_skeleton_animation(skel_frames, fps=30):
    """將片段的 3D 骨架陣列繪製為動態對照的 HTML 影片"""
    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='3d')
    
    # 近似的 17 關節連線定義 (COCO/SMPL 風格)
    # 根據常見資料，若 0 是骨盆：
    bones = [
        (0, 1), (1, 2), (2, 3),        # 右腿
        (0, 4), (4, 5), (5, 6),        # 左腿
        (0, 7), (7, 8), (8, 9), (9, 10), # 軀幹與頭部
        (8, 11), (11, 12), (12, 13),   # 左手
        (8, 14), (14, 15), (15, 16)    # 右手
    ]
    
    lines = [ax.plot([], [], [], c='blue', lw=2)[0] for _ in bones]
    scat = ax.scatter([], [], [], c='red', s=20, alpha=0.5)
    
    # 設定適當的 3D 範圍
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([0, 2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    # 設定視角
    ax.view_init(elev=10, azim=0)
    plt.close(fig) # 隱藏靜態圖表
    
    def update(frame_idx):
        pts = skel_frames[frame_idx]
        scat._offsets3d = (pts[:,0], pts[:,1], pts[:,2])
        for line, bone in zip(lines, bones):
            p1, p2 = pts[bone[0]], pts[bone[1]]
            line.set_data([p1[0], p2[0]], [p1[1], p2[1]])
            line.set_3d_properties([p1[2], p2[2]])
        return lines + [scat]
    
    anim = animation.FuncAnimation(fig, update, frames=len(skel_frames), interval=1000/fps, blit=False)
    return HTML(anim.to_jshtml())

print("骨架動畫繪製邏輯準備完成！")

骨架動畫繪製邏輯準備完成！


In [8]:
# ========================
# Streaming Multi-Agent Analysis (Interactive Bot Version)
# ========================
import time

total_segments = len(segments_df)

for i, row in segments_df.iterrows():
    clear_output(wait=True)
    
    # 1. 產生 AI Sees 描述
    sees_text, keywords = ai_sees(row, cultural_library)
    
    # 2. 準備廣播資料
    seg_metrics = {
        "timestamp_sec": row['timestamp_sec'],
        "energy": round(row['energy'], 2),
        "torque": round(row['torque'], 2),
        "sees_content": sees_text,
        "keywords": keywords
    }
    
    # 3. 發送廣播請求至 Discord Bot 控制器
    state_manager.request_broadcast(seg_metrics, mode="bajiajiang")
    
    # 4. 在 Notebook 顯示進度
    print("="*60)
    print(f"處理進度: {i+1}/{total_segments} | 時間: {row['timestamp_sec']} 秒")
    print("-"*30)
    print(sees_text)
    print("-"*30)
    print(f"[AI Reasoning] Keywords: {keywords}")
    print("\n[Discord Status] 等待機器人對話中... (請至 Discord 頻道查看)")
    
    # 5. 繪製骨架動畫
    start_f = int(row['frame_start'])
    end_f = int(min(start_f + INTERVAL_FRAMES, len(skel_data)))
    display(create_skeleton_animation(skel_data[start_f:end_f]))
    
    # 6. 阻塞 Notebook 直到 Discord 上的對話結束
    while not state_manager.should_proceed():
        time.sleep(2)
    
print("\n✅ 所有片段分析與 Discord 互動廣播已完成！")

處理進度: 15/15 | 時間: 70.0 秒
------------------------------
【AI sees】 在此次舞動中，能量指標為2.92，顯示出舞者在動作中展現出中等的活力與動態表現。舞者的體積為0.0562，表明其動作範圍相對集中，可能偏向於精緻的細節展現。扭矩數值6.40顯示出舞者在轉動或變換姿勢時，具備一定的力量與控制能力，這使得舞者能夠在動作中保持穩定性與力量感。高達33382.73的震動指數（Jerk）則暗示了舞者在動作變化中的急劇度，可能涉及快速的轉換或突變動作，這在舞蹈中常見於強烈的情感表達或劇烈的動作衝擊。
------------------------------
【AI says - 刑具爺】 舞者之動作，須如刑具之運用，精準、穩定且具威嚴，方能震撼人心！
【AI says - 甘柳將軍】 這舞者如狂風怒吼，扭轉之際，邪祟無所遁形！
【AI says - 廟口長老】 藏氣於舞，靈動生風。

[AI Reasoning & Traceability]
- Linked Metrics: Energy=2.92, Torque=6.40
- Keywords from Library: 能量, 體積, 扭矩, 震動, 女子八家將, 剛柔並濟, 當代家將靈魂
- Source: ba_jia_jiang_cultural_library.json

[動態骨架對照]:



✨ 全部分析完成！正在匯出檔案...
📁 已輸出：bajiajiang_multi_agent_results.json
📁 已輸出：bajiajiang_ai_says_multi_agents.json
